# 03 · 剪枝对比（E2 / E3 / E5）

所有方法从**同一个 Dense-Aug checkpoint** 出发，逐层非结构化剪枝（每层相同稀疏度），用同一批打分原图、同一批评估对和固定遮挡。

| 方法 | 打分数据 | 反传次数 |
|---|---|---|
| Random | 随机分数（按打分种子） | 0 |
| Magnitude | \|w\| | 0 |
| Clean-Taylor | B 张干净图 | B |
| Occ-Taylor | B 张遮挡图 | B |
| Mixed-Taylor | B/2 干净 + B/2 遮挡，直接平均 | B |
| OAGP | 与 Mixed 相同的 B 个视图；每层按均值归一化后加权平均（α） | B |
| OAGP-max | 同上，逐权重取较大者 | B |

**共享校准**：B=1024 张训练原图分成 A、B 两半，一次性算 4 组逐样本 Taylor 分数（A 干净、B 干净、A 遮挡、B 遮挡，共 2048 次反传，约 30 秒）。Clean = A干净+B干净，Occ = A遮挡+B遮挡，Mixed / OAGP / OAGP-max 都用 A干净 + B遮挡 —— 三者看到的视图完全相同，只差合并方式。

**流程**：先只在 CASIA 留出验证对上跑（`TEST_SETS = ()`）选稀疏度和 α；设置冻结后再加 `("lfw",)` 得到最终测试结果。微调用独立的小学习率（默认 0.01，cosine 到 0），每步后重新施加掩码。

In [ ]:
from pathlib import Path
import os, sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/occprune").is_dir())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
from occprune.prune_run import run_pruning, METHODS
from occprune.prune_report import load_runs, metrics_table, summary_table, paired_bootstrap
print("methods:", METHODS)

## 1. 设置
- `TAG` 区分一组实验（输出在 `results/pruning/runs/<TAG>_score<seed>/`，已存在的目录不会被覆盖，会直接读取）。
- 默认设置就是已经跑完的 **E2 一次性剪枝网格**：7 种方法 × 50–90% × 打分种子 42/43/44，直接运行即可读出结果，不需要 GPU。
- 其他已完成的实验，把下面一行设置换掉即可读取：

| 实验 | `TAG` | `METHODS_RUN` | `SPARSITIES` | `FT_EPOCHS` | 其他 |
|---|---|---|---|---|---|
| E2 一次性网格 | `"oneshot_grid"` | `list(METHODS)` | `[0.5, 0.6, 0.7, 0.8, 0.9]` | 0 | |
| E3 微调 | `"finetune_grid"` | `list(METHODS)` | `[0.6, 0.7]` | 1 | |
| E6 α 扫描 | `"alpha0.75"`（或 `alpha0`、`alpha0.25`、`alpha1`） | `["oagp"]` | `[0.5, 0.6, 0.7]` | 0 | `ALPHA = 0.75` |

- 新实验：先只在 CASIA 留出验证对上跑（`TEST_SETS = ()`）选设置，冻结后再加 `("lfw",)`。正式结果统一在一台机器上跑（不同 GPU / MPS 之间约有 0.1–0.5 个百分点的差异）。

In [ ]:
CHECKPOINT = "checkpoints/dense_aug_seed42.pt"
TAG = "oneshot_grid"
METHODS_RUN = list(METHODS)            # E6: ["oagp"]
SPARSITIES = [0.5, 0.6, 0.7, 0.8, 0.9]
SCORING_SEEDS = [42, 43, 44]
FT_EPOCHS = 0                          # E3 微调: 1
ALPHA = None                           # None = configs/default.yaml 里的 0.5
TEST_SETS = ("lfw",)                   # 选设置时用 ()（只看 CASIA 验证对）
RUN = False                            # True = 运行缺失的实验（需要 checkpoint 和数据缓存）

run_dirs = []
for seed in SCORING_SEEDS:
    dest = Path("results/pruning/runs") / f"{TAG}_score{seed}"
    run_dirs.append(str(dest))
    if (dest / "result.json").is_file():
        print("exists:", dest)
    elif RUN:
        run_pruning(CHECKPOINT, dest, methods=METHODS_RUN, sparsities=SPARSITIES, scoring_seed=seed,
                    alpha=ALPHA, ft_epochs=FT_EPOCHS, test_sets=TEST_SETS)
    else:
        print("missing (set RUN = True):", dest)

## 2. 结果表
准确率（%）按打分种子取平均；`region_mean` 是双眼、口鼻、上半脸、下半脸四种区域遮挡的平均。`dense` 行是未剪枝的参照。

In [ ]:
existing = [d for d in run_dirs if (Path(d) / "result.json").is_file()]
DATASET = "lfw" if TEST_SETS else "validation"
if existing:
    df = metrics_table(load_runs(existing))
    display(summary_table(df, dataset=DATASET, stage="one_shot").round(2))
else:
    df = None
    print("no finished runs yet")

In [ ]:
# 微调后的结果（FT_EPOCHS > 0 时）
if df is not None and (df.stage == "fine_tuned").any():
    display(summary_table(df, dataset=DATASET, stage="fine_tuned").round(2))

## 3. 两个方法的配对比较
同一批评估对上，各自用 10 折阈值判断每一对，再对**对**重采样，得到准确率差（B − A）的 95% 置信区间。区间跨 0 时不下结论。

In [ ]:
if existing and {"oagp", "oagp_max"} <= set(METHODS_RUN):
    display(paired_bootstrap(existing[0], "oagp", "oagp_max", sparsity=SPARSITIES[0], dataset=DATASET).round(2))

## 4. 已完成实验的报告（直接读取 `results/pruning/`）

下面三份报告由 `prune_report` 生成，每个文件夹里的 README 写了结论、数字含义和复现命令：

- [一次性剪枝网格](../results/pruning/oneshot_grid/README.md)：E2、E4、E5、E7。不微调时 Magnitude 最稳；Occ-Taylor 明显好于 Clean-Taylor；干净与遮挡重要性 Spearman 0.96。
- [剪枝 + 微调](../results/pruning/finetune_grid/README.md)：E3。微调 1 epoch 后，除 Random 外都恢复到 95.0–95.4%。
- [OAGP α 扫描](../results/pruning/alpha_sweep/README.md)：E6。α 越偏向遮挡分数越好，验证集选出 0.75–1。

表中是 3 个打分种子的 mean ± sd（%）；"Mean of four held-out regions" 是双眼、口鼻、上半脸、下半脸的平均。

In [ ]:
from IPython.display import Image, Markdown, display

REPORTS = Path("results/pruning")
display(Markdown("### LFW, one-shot\n" + (REPORTS / "oneshot_grid/summary_lfw_one_shot.md").read_text(encoding="utf-8")))
display(Image(str(REPORTS / "oneshot_grid/sparsity_lfw_one_shot.png"), width=1000))
display(Image(str(REPORTS / "oneshot_grid/bank_agreement.png"), width=900))

In [ ]:
display(Markdown("### LFW, after 1 epoch of fine-tuning\n" + (REPORTS / "finetune_grid/summary_lfw_fine_tuned.md").read_text(encoding="utf-8")))
display(Image(str(REPORTS / "finetune_grid/sparsity_lfw_fine_tuned.png"), width=1000))
display(Markdown((REPORTS / "alpha_sweep/alpha_sweep.md").read_text(encoding="utf-8")))
display(Image(str(REPORTS / "alpha_sweep/alpha_sweep.png"), width=900))

重新生成报告（跑了新的种子或稀疏度之后）：把 `REBUILD` 改成 `True`。E4 的重要性一致性需要本地的 `condition_scores.npz`（不进 git），没有时会自动跳过。

In [ ]:
REBUILD = False
if REBUILD:
    import glob
    from occprune.prune_report import write_alpha_report, write_grid_report

    runs = lambda tag: sorted(glob.glob(f"results/pruning/runs/{tag}_score*"))
    write_grid_report(runs("oneshot_grid"), REPORTS / "oneshot_grid")
    write_grid_report(runs("finetune_grid"), REPORTS / "finetune_grid")
    write_alpha_report(sorted(glob.glob("results/pruning/runs/alpha*_score*")) + runs("oneshot_grid"),
                       REPORTS / "alpha_sweep")